In [ ]:
"""
Objetivo: Conectar à API do Federal Reserve (FRED), extrair séries macroeconômicas mensais 
(TED Spread, Desemprego e Fed Funds) e exportar a base consolidada para a etapa de 
estimações dinâmicas via Rolling-Window VAR focada no período da crise de 2008.
"""

import pandas as pd
from fredapi import Fred
import warnings
import os
os.environ['FRED_API_KEY'] = 'FRED_API_KEY'
import sys

warnings.filterwarnings('ignore')

# ==========================================
# 1. AUTENTICAÇÃO E CONEXÃO COM O FRED
# ==========================================
# A chave de API deve ser configurada no ambiente (ex: export FRED_API_KEY='sua_chave')
# ou inserida no arquivo .env para evitar vazamento em repositórios públicos.
API_KEY = os.getenv('FRED_API_KEY')

if not API_KEY:
    print("[ERRO] Chave da API do FRED ausente.")
    print("-> Configure a variável de ambiente 'FRED_API_KEY' antes de rodar o script.")
    sys.exit(1)

def extrair_painel_rolling_var(chave_api: str, arquivo_saida: str):
    """Extrai, alinha as frequências mensais e exporta o painel macroeconômico."""
    print("[*] Autenticando no servidor do Federal Reserve (FRED)...")
    
    try:
        fred = Fred(api_key=chave_api)
        
        # Janela delimitada para capturar calmaria, Subprime e Pós-Lehman
        data_inicio, data_fim = '2000-01-01', '2015-12-31'
        
        print(f"[*] Extraindo séries ({data_inicio} a {data_fim}) em frequência mensal...")
        
        # TED Spread: Coleta dados diários e converte para média mensal
        ted_spread = fred.get_series('TEDRATE', observation_start=data_inicio, observation_end=data_fim)
        ted_spread = ted_spread.resample('MS').mean()
        
        # Desemprego e Fed Funds (Mensais)
        unrate = fred.get_series('UNRATE', observation_start=data_inicio, observation_end=data_fim)
        fedfunds = fred.get_series('FEDFUNDS', observation_start=data_inicio, observation_end=data_fim)
        
        # Consolidação do Vetor Macro
        df_var = pd.DataFrame({
            'TED_Spread': ted_spread,
            'Desemprego': unrate,
            'Fed_Funds': fedfunds
        })
        
        # Limpeza e alinhamento de calendário
        df_var.index.name = 'Data'
        df_var.dropna(inplace=True)
        
        # Exportação
        df_var.to_csv(arquivo_saida)
        
        print("\n[SUCESSO] Base do laboratório Rolling-Window VAR estruturada e exportada.")
        print(f"[*] Arquivo gerado: {arquivo_saida}")
        print(f"[*] Dimensões: {df_var.shape[0]} meses capturados.")
        
    except ValueError as ve:
        print(f"\n[ERRO DE AUTENTICAÇÃO] {ve}. Verifique sua chave de API.")
    except Exception as e:
        print(f"\n[ERRO CRÍTICO] Falha na comunicação com o FRED: {e}")

if __name__ == "__main__":
    extrair_painel_rolling_var(API_KEY, '07_base_rolling_var_fred.csv')